<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/Mx_Oscar_Hdez_Hands_On_WhatsApp_Cloud_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: WHATSAPP CLOUD API**

Una vez vista la masterclass ***WhatsApp Cloud API***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python***  y el modelo de ***llama 3.1*** de Meta.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/19F84m5PKCMKEOHxL7z3MUqN98HEhwLWS?usp=drive_link).

## **LOGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [25]:
# ============================================================
# LEER CREDENCIALES DESDE COLAB SECRETS
# ============================================================

from google.colab import userdata

WHATSAPP_TOKEN = userdata.get("WHATSAPP_TOKEN")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")

print("WHATSAPP_TOKEN:   ✓ cargado")
print("NGROK_AUTHTOKEN:  ✓ cargado")
print("VERIFY_TOKEN:     ✓ cargado")
print("PHONE_NUMBER_ID:  ✓ cargado")

WHATSAPP_TOKEN:   ✓ cargado
NGROK_AUTHTOKEN:  ✓ cargado
VERIFY_TOKEN:     ✓ cargado
PHONE_NUMBER_ID:  ✓ cargado


#### **INSTALACIÓN Y ARRANQUE DE OLLAMA**

Ollama no viene preinstalado en la máquina virtual de Google Colab, por lo que se debe instalar y arrancar en segundo plano al inicio de cada sesión.

In [2]:
# Instalar Ollama
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (869 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122809 files and director

In [3]:
# Iniciar el servidor de Ollama
import subprocess
import time

# Iniciar Ollama en segundo plano
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama iniciado")

# ============================================================
# DESCARGAR MODELO LLAMA 3.1
# ============================================================

!ollama pull llama3.1

# ============================================================
# VERIFICAR MODELOS INSTALADOS
# ============================================================

!ollama list

# ============================================================
# PROBAR LLAMA 3.1
# ============================================================

import requests

respuesta = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1",
        "prompt": "Responde únicamente: Llama 3.1 funcionando correctamente.",
        "stream": False
    },
    timeout=120
)

print(respuesta.json()["response"])

Ollama iniciado

NAME               ID              SIZE      MODIFIED               
llama3.1:latest    46e0c10c039e    4.9 GB    Less than a second ago    
Sí, la Llama 3.1 funciona correctamente.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

Con Ollama corriendo, se arma la lógica del asistente: la estructura del servidor con sus registros en memoria, la normalización de números mexicanos (necesaria para que la API de envío acepte el número que entrega el webhook), y la función que genera la respuesta real con Llama 3.1.

In [4]:
# ============================================================
# CONFIGURAR EL SERVIDOR DEL WEBHOOK
# ============================================================

!pip install -q fastapi uvicorn
from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import PlainTextResponse
import requests
import time

# Crear aplicación FastAPI
app = FastAPI()

# ------------------------------------------------------------
# Configuración de Ollama
# ------------------------------------------------------------

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.1"

# Bitácora de payloads recibidos
mensajes_recibidos = []

# Usuarios que ya recibieron el saludo inicial
usuarios_conocidos = set()

print("✓ Servidor FastAPI configurado")
print("✓ Modelo:", OLLAMA_MODEL)

✓ Servidor FastAPI configurado
✓ Modelo: llama3.1


In [5]:
# ============================================================
# DEFINIR FUNCIÓN DE NORMALIZACIÓN DE NÚMERO
# ============================================================

def normalizar_numero_mx(numero):

    if not numero:
        return None

    numero = "".join(filter(str.isdigit, str(numero)))

    # Formato antiguo móvil mexicano: 521XXXXXXXXXX
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    # Número nacional de 10 dígitos
    elif len(numero) == 10:
        numero = "52" + numero

    return numero

In [6]:
# ============================================================
# DEFINIR FUNCIÓN DE GENERACIÓN DE RESPUESTA
# ============================================================

def generar_respuesta(mensaje):

    prompt = f"""
Eres un asistente virtual de pedidos que atiende usuarios por WhatsApp.

Reglas:
- Responde siempre de manera amable y profesional.
- Sé claro y directo.
- La respuesta debe tener máximo 2 o 3 líneas.
- No des explicaciones innecesarias.
- No inventes información.
- Si no tienes información suficiente, pregunta al usuario.

Mensaje del usuario:
{mensaje}

Respuesta:
"""

    try:

        response = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )

        response.raise_for_status()

        return response.json()["response"].strip()

    except Exception as e:

        print("❌ Error con Ollama:", e)

        return (
            "Lo siento, no pude procesar tu solicitud "
            "en este momento."
        )

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante.

La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta, evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [7]:
# ============================================================
# DEFINIR ENDPOINT DE VERIFICACIÓN DE CREDENCIALES
# ============================================================

@app.get("/webhook")
async def verificar_webhook(request: Request):

    mode = request.query_params.get("hub.mode")
    token = request.query_params.get("hub.verify_token")
    challenge = request.query_params.get("hub.challenge")

    if mode == "subscribe" and token == VERIFY_TOKEN:

        print("✓ Webhook verificado correctamente")

        return PlainTextResponse(
            content=challenge,
            status_code=200
        )

    return PlainTextResponse(
        content="Forbidden",
        status_code=403
    )

In [8]:
# ============================================================
# DEFINIR ENDPOINT DE RECEPCIÓN DE MENSAJES
# ============================================================

@app.post("/webhook")
async def recibir_webhook(
    request: Request,
    background_tasks: BackgroundTasks
):

    try:

        data = await request.json()

        # Guardar payload completo en la bitácora
        mensajes_recibidos.append(data)

        entry = data.get("entry", [])

        if not entry:
            return {"status": "ok"}

        changes = entry[0].get("changes", [])

        if not changes:
            return {"status": "ok"}

        value = changes[0].get("value", {})

        messages = value.get("messages", [])

        # Puede ser una notificación de estado
        if not messages:
            return {"status": "ok"}

        mensaje = messages[0]

        numero = mensaje.get("from")

        # Por ahora solo procesamos texto
        if mensaje.get("type") != "text":
            return {"status": "ignored"}

        texto = (
            mensaje
            .get("text", {})
            .get("body", "")
            .strip()
        )

        print("📩 Mensaje recibido")
        print("De:", numero)
        print("Texto:", texto)

        # Procesamiento en segundo plano
        background_tasks.add_task(
            procesar_mensaje,
            numero,
            texto
        )

        # Respuesta inmediata a Meta
        return {"status": "accepted"}

    except Exception as e:

        print("❌ Error en webhook:", e)

        return {"status": "error"}

In [9]:
# ============================================================
# DEFINIR FUNCIÓN DE ENVÍO DE RESPUESTA
# ============================================================

def enviar_respuesta(numero, texto):

    numero = normalizar_numero_mx(numero)

    url = (
        f"https://graph.facebook.com/v26.0/"
        f"{PHONE_NUMBER_ID}/messages"
    )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {
            "body": texto
        }
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        if response.ok:
            print("✓ Respuesta enviada a:", numero)
            return True

        print("❌ Error enviando respuesta")
        print(response.status_code)
        print(response.text)

        return False

    except Exception as e:

        print("❌ Error:", e)
        return False

In [10]:
# ============================================================
# DEFINIR FUNCIÓN DE PROCESAMIENTO DE RESPUESTA
# ============================================================

def procesar_mensaje(numero, texto):

    numero = normalizar_numero_mx(numero)

    print("⚙ Procesando mensaje de:", numero)

    # --------------------------------------------------------
    # USUARIO NUEVO
    # --------------------------------------------------------

    if numero not in usuarios_conocidos:

        saludo = (
            "¡Hola! 👋 Bienvenido a nuestro asistente de pedidos. "
            "¿En qué podemos ayudarte hoy?"
        )

        enviar_respuesta(
            numero,
            saludo
        )

        # Registrar usuario
        usuarios_conocidos.add(numero)

        print("✓ Usuario agregado a usuarios_conocidos")

        return

    # --------------------------------------------------------
    # USUARIO CONOCIDO → LLAMA
    # --------------------------------------------------------

    respuesta = generar_respuesta(texto)

    print("🤖 Llama:", respuesta)

    enviar_respuesta(
        numero,
        respuesta
    )

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers.

\begin{equation} WhatsApp  \, \, \, \Rightarrow \, \, \, Configuración  \, \,de  \, \,la  \, \,API  \, \, \, \Rightarrow \, \, \, Configurar  \, \,webhooks \end{equation}

In [11]:
# ============================================================
# INICIAR SERVIDOR FASTAPI CON UVICORN
# ============================================================

import uvicorn
import threading
import time

def iniciar_servidor():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=iniciar_servidor,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("✓ Servidor FastAPI iniciado en puerto 8000")

INFO:     Started server process [2871]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✓ Servidor FastAPI iniciado en puerto 8000


In [12]:
# ============================================================
# INSTALAR E INICIAR TÚNEL PÚBLICO CON NGROK
# ============================================================

!pip install -q pyngrok

from pyngrok import ngrok
from google.colab import userdata

# ------------------------------------------------------------
# Leer token de ngrok desde Colab Secrets
# ------------------------------------------------------------

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

if not NGROK_AUTHTOKEN:
    raise ValueError(
        "❌ No se encontró NGROK_AUTHTOKEN en Colab Secrets"
    )

# Configurar autenticación de ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# ------------------------------------------------------------
# Cerrar túneles anteriores
# ------------------------------------------------------------

ngrok.kill()

# ------------------------------------------------------------
# Crear túnel HTTPS hacia FastAPI
# ------------------------------------------------------------

tunnel = ngrok.connect(8000)

PUBLIC_URL = tunnel.public_url

# Asegurarnos de utilizar HTTPS
if PUBLIC_URL.startswith("http://"):
    PUBLIC_URL = PUBLIC_URL.replace(
        "http://",
        "https://",
        1
    )

print("✓ Túnel ngrok iniciado")
print("✓ URL pública:", PUBLIC_URL)
print("✓ Webhook:", f"{PUBLIC_URL}/webhook")


✓ Túnel ngrok iniciado
✓ URL pública: https://unneeded-playhouse-tuition.ngrok-free.dev
✓ Webhook: https://unneeded-playhouse-tuition.ngrok-free.dev/webhook


In [13]:
# ============================================================
# BITÁCORA DEL ASISTENTE
# ============================================================

from datetime import datetime

LOGS = []

def log(mensaje):
    hora = datetime.now().strftime("%H:%M:%S")
    linea = f"[{hora}] {mensaje}"

    LOGS.append(linea)
    print(linea)

print("✓ Bitácora inicializada")

✓ Bitácora inicializada


In [16]:
# ============================================================
# MOSTRAR BITÁCORA
# ============================================================

print("\n".join(LOGS[-50:]))

In [17]:
# ============================================================
# DIAGNÓSTICO DEL WEBHOOK ACTUAL
# ============================================================

from pyngrok import ngrok
import requests

print("=== TÚNELES ACTIVOS ===")

tunnels = ngrok.get_tunnels()

for tunnel in tunnels:
    print("URL:", tunnel.public_url)
    print("Destino:", tunnel.config.get("addr"))

print("\n=== ENDPOINTS FASTAPI ===")

for route in app.routes:
    print(
        route.path,
        getattr(route, "methods", None)
    )

print("\n=== PRUEBA LOCAL ===")

r = requests.get(
    "http://localhost:8000/webhook",
    params={
        "hub.mode": "subscribe",
        "hub.verify_token": VERIFY_TOKEN,
        "hub.challenge": "999999"
    },
    timeout=10
)

print("Local:", r.status_code, r.text)

print("\n=== URL QUE DEBE ESTAR EN META ===")

if tunnels:
    PUBLIC_URL_ACTUAL = tunnels[0].public_url
    print(f"{PUBLIC_URL_ACTUAL}/webhook")
else:
    print("❌ No existe ningún túnel ngrok activo")

=== TÚNELES ACTIVOS ===
URL: https://unneeded-playhouse-tuition.ngrok-free.dev
Destino: http://localhost:8000

=== ENDPOINTS FASTAPI ===
/openapi.json {'GET', 'HEAD'}
/docs {'GET', 'HEAD'}
/docs/oauth2-redirect {'GET', 'HEAD'}
/redoc {'GET', 'HEAD'}
/webhook {'GET'}
/webhook {'POST'}

=== PRUEBA LOCAL ===
✓ Webhook verificado correctamente
INFO:     127.0.0.1:45638 - "GET /webhook?hub.mode=subscribe&hub.verify_token=tesoem_whatsapp_2026&hub.challenge=999999 HTTP/1.1" 200 OK
Local: 200 999999

=== URL QUE DEBE ESTAR EN META ===
https://unneeded-playhouse-tuition.ngrok-free.dev/webhook


In [34]:
print("Payloads recibidos:", len(mensajes_recibidos))

import json

if mensajes_recibidos:
    print(json.dumps(
        mensajes_recibidos[-1],
        indent=2,
        ensure_ascii=False
    ))

Payloads recibidos: 9
{
  "object": "whatsapp_business_account",
  "entry": [
    {
      "id": "1389496259965043",
      "changes": [
        {
          "value": {
            "messaging_product": "whatsapp",
            "metadata": {
              "display_phone_number": "15551699143",
              "phone_number_id": "1321115231092474"
            },
            "contacts": [
              {
                "profile": {
                  "name": "Eliot Max"
                },
                "wa_id": "5215536699933",
                "user_id": "MX.1563261751784479"
              }
            ],
            "messages": [
              {
                "from": "5215536699933",
                "from_user_id": "MX.1563261751784479",
                "id": "wamid.HBgNNTIxNTUzNjY5OTkzMxUCABIYIEE1Rjk5RDk1NzFEQjcyNTRBODdBQjBCNEUwODlFNTEzAA==",
                "timestamp": "1790100450",
                "text": {
                  "body": "Quiero hacer un pedido de pizza"
                },

### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje para confirmar que el saludo inicial funciona, y un segundo mensaje para confirmar que ya se recibe una respuesta generada por Llama.

In [ ]:
# Mandar mensaje al bot para recibir un saludo
Hola

In [ ]:
# Seguir conversando con el bot para interactuar con Llama
Quiero hacer un pedido de pizza

## **CHALLENGE: ASISTENTE DE PEDIDOS**

Una vez visto el ***Hands-On: Hands-On: WhatsApp Cloud API***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

A partir del asistente de pedidos construido, se agregan dos mejoras:

* Un mensaje de bienvenida para usuarios nuevos.
* Un tono de respuesta más controlado.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab (no viene preinstalado en la máquina virtual, así que se instala y se arranca en segundo plano al inicio de la sesión), descarga el modelo Llama 3.1 8B, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [2]:
# Instalar Ollama
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (497 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directo

In [3]:
# Iniciar el servidor de Ollama
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

!ollama pull llama3.1

In [4]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

WHATSAPP_TOKEN = userdata.get("WHATSAPP_TOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")


**2. Configura el servidor y el registro de usuarios conocidos:**

   * Crea el servidor FastAPI con las estructuras en memoria necesarias: `mensajes_recibidos` para la bitácora de payloads, y `usuarios_conocidos` (un `set`) para llevar registro de quién ya recibió el saludo.
   * Define la función `normalizar_numero_mx` para que los números mexicanos se envíen en el formato correcto.

In [5]:
# Configurar el servidor del webhook
from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import PlainTextResponse
import requests

app = FastAPI()
mensajes_recibidos = []
usuarios_conocidos = set()

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.1"

In [6]:
# Definir función de normalización de número
def normalizar_numero_mx(numero):
    numero = "".join(filter(str.isdigit, str(numero)))
    return "52" + numero[3:] if numero.startswith("521") else numero

**3. Genera la respuesta con un tono controlado:** Ajusta el prompt que se envía a Llama para que las respuestas se mantengan breves (máximo 2-3 líneas) y con un tono amable.

In [7]:
# Definir función de generación de respuesta
def generar_respuesta(mensaje):
    prompt = f"""Responde de forma amable, clara y en máximo 2-3 líneas.
No inventes información.

Usuario: {mensaje}
Asistente:"""

    r = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False
    })

    return r.json()["response"].strip()

**4. Define los endpoints del webhook:**

   * Crea el endpoint de verificación (`GET /webhook`).
   * Crea el endpoint de recepción de mensajes (`POST /webhook`), delegando el trabajo pesado a `BackgroundTasks` para responder de inmediato a Meta sin esperar a Ollama.

In [8]:
# Definir endpoint de verificación de credenciales
@app.get("/webhook")
async def verificar(request: Request):
    q = request.query_params
    if q.get("hub.mode") == "subscribe" and q.get("hub.verify_token") == VERIFY_TOKEN:
        return PlainTextResponse(q.get("hub.challenge"))
    return PlainTextResponse("Forbidden", status_code=403)

In [9]:
# Definir endpoint de recepción de mensajes
@app.post("/webhook")
async def recibir(request: Request, background_tasks: BackgroundTasks):
    data = await request.json()
    mensajes_recibidos.append(data)

    try:
        mensaje = data["entry"][0]["changes"][0]["value"]["messages"][0]
        if mensaje["type"] == "text":
            background_tasks.add_task(
                procesar_mensaje,
                mensaje["from"],
                mensaje["text"]["body"]
            )
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}


**5. Agrega el saludo inicial y el envío de la respuesta:**

   * Define la función que envía el mensaje de WhatsApp.
   * Define la función que decide si se manda el saludo fijo o la respuesta generada por Llama, según si el número ya está en `usuarios_conocidos`.

In [10]:
# Definir función de envío de respuesta
def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_NUMBER_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
    data = {
        "messaging_product": "whatsapp",
        "to": normalizar_numero_mx(numero),
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=data)
    print("✓ Enviado" if r.ok else f"❌ Error: {r.text}")

In [11]:
# Definir función de procesamiento de respuesta
def procesar_mensaje(numero, texto):
    numero = normalizar_numero_mx(numero)

    if numero not in usuarios_conocidos:
        respuesta = "¡Hola! 👋 Bienvenido a nuestro asistente de pedidos. ¿En qué podemos ayudarte?"
        usuarios_conocidos.add(numero)
    else:
        respuesta = generar_respuesta(texto)

    enviar_respuesta(numero, respuesta)

**6. Expón el servidor y prueba el flujo completo:**

   * Instala `pyngrok`, abre el túnel público y arranca el servidor en un hilo aparte.
   * Manda un primer mensaje de prueba y confirma que se recibe el saludo; manda un segundo mensaje y confirma que ya llega la respuesta generada por Llama.

In [12]:
# Instalar e iniciar túnel público con ngrok
!pip install -q pyngrok uvicorn

import threading, uvicorn
from pyngrok import ngrok

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True
).start()

ngrok.set_auth_token(NGROK_AUTHTOKEN)
PUBLIC_URL = ngrok.connect(8000).public_url

print("Webhook:", f"{PUBLIC_URL}/webhook")

INFO:     Started server process [3470]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Webhook: https://unneeded-playhouse-tuition.ngrok-free.dev/webhook


In [ ]:
# Mandar mensaje al bot para recibir un saludo
# En WhatsApp enviar: Hola

In [ ]:
# Seguir conversando con el bot para interactuar con Llama
# En WhatsApp enviar un segundo mensaje, por ejemplo:
# Quiero hacer un pedido de dos pizzas.